# 11-2절 연습 문제 풀이

이 노트북은 11-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch11/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 11장 공통 - Fashion-MNIST VAE / DCGAN 도우미
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
DATA_ROOT = '../../download'
LATENT_DIM = 16

def fashion_loader(batch_size=128, normalize_tanh=False):
    tf = [transforms.ToTensor()]
    if normalize_tanh: tf.append(transforms.Normalize((0.5,), (0.5,)))
    ds = datasets.FashionMNIST(root=DATA_ROOT, train=True, download=True,
                               transform=transforms.Compose(tf))
    return DataLoader(ds, batch_size=batch_size, shuffle=True)

class FashionVAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU())
        self.flatten_dim = 64 * 7 * 7
        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_log_var = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_z_to_fmap = nn.Linear(latent_dim, self.flatten_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, 2, 1, output_padding=1), nn.Sigmoid())
    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_log_var(h)
    def reparameterize(self, mu, log_var):
        if self.training:
            std = torch.exp(0.5 * log_var)
            return mu + torch.randn_like(std) * std
        return mu
    def decode(self, z):
        return self.decoder(self.fc_z_to_fmap(z).view(-1, 64, 7, 7))
    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var

bce_loss = nn.BCELoss(reduction='sum')
def vae_loss(recon, x, mu, log_var):
    bce = bce_loss(recon, x) / x.size(0)
    kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    return bce + kl, bce, kl

def train_vae(model, epochs=10, lr=1e-3, latent_dim=16):
    loader = fashion_loader()
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for e in range(1, epochs + 1):
        model.train(); tot = n = 0
        for x, _ in loader:
            x = x.to(device)
            recon, mu, log_var = model(x)
            loss, bce, kl = vae_loss(recon, x, mu, log_var)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(x); n += len(x)
        if e % 5 == 0 or e == 1: print(f'  {e}/{epochs} 손실 {tot / n:.2f}')
    return model

class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.latent_dim = latent_dim
        self.fc_z_to_fmap = nn.Sequential(
            nn.Linear(latent_dim, 256 * 7 * 7), nn.BatchNorm1d(256 * 7 * 7), nn.ReLU())
        self.gen_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 4, 2, 1), nn.Tanh())
    def forward(self, z):
        return self.gen_conv(self.fc_z_to_fmap(z).view(-1, 256, 7, 7))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.disc_conv = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128 * 7 * 7, 1), nn.Sigmoid())
    def forward(self, x): return self.fc(self.disc_conv(x))

def weights_init(m):
    name = m.__class__.__name__
    if 'Conv' in name: nn.init.normal_(m.weight, 0.0, 0.02)
    elif 'BatchNorm' in name:
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.constant_(m.bias, 0)

def train_dcgan(latent_dim=100, epochs=10, detach_g=False, no_detach_d=False):
    loader = fashion_loader(normalize_tanh=True)
    torch.manual_seed(SEED)
    G, D = Generator(latent_dim).to(device), Discriminator().to(device)
    G.apply(weights_init); D.apply(weights_init)
    crit = nn.BCELoss()
    optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    for e in range(1, epochs + 1):
        tl_d = tl_g = n = 0
        for real, _ in loader:
            real = real.to(device); B = real.size(0)
            ones, zeros = torch.ones(B, 1, device=device), torch.zeros(B, 1, device=device)
            optD.zero_grad()
            z = torch.randn(B, latent_dim, device=device)
            fake = G(z)
            loss_d = crit(D(real), ones) + crit(
                D(fake if no_detach_d else fake.detach()), zeros)
            loss_d.backward(); optD.step()
            optG.zero_grad()
            z = torch.randn(B, latent_dim, device=device)
            fake = G(z)
            loss_g = crit(D(fake.detach() if detach_g else fake), ones)
            loss_g.backward(); optG.step()
            tl_d += loss_d.item() * B; tl_g += loss_g.item() * B; n += B
        if e % 5 == 0 or e == 1:
            print(f'  {e}/{epochs} D {tl_d / n / 2:.4f} / G {tl_g / n:.4f}')
    return G, D

## 연습 11-5

detach() 사용 위치를 두 경우로 바꿔 실험해 보자.

[코드 11-12]의 생성자 학습 단계에서 D(fake_images)를 D(fake_images.detach())로 바꾸면 생성자 학습이 어떻게 달라지는가?

반대로 [코드 11-12]의 판별자 학습 단계에서 .detach()를 제거하면 어떤 변화가 생기는가? 두 신경망의 기울기 흐름과 함께 정리해 보자.

In [ ]:
print('[1) 생성자 학습에서 detach() 사용]')
G1, _ = train_dcgan(epochs=5, detach_g=True)
print('\n[2) 판별자 학습에서 detach() 제거]')
G2, _ = train_dcgan(epochs=5, no_detach_d=True)

**1) 생성자 학습에 `detach()`를 쓰면**: 손실이 생성자까지 역전파되지 않아 **생성자가 전혀 학습되지 않는다**. 생성자 손실은 계속 높은 값에 머물고 생성 이미지는 잡음 그대로다.

**2) 판별자 학습에서 `detach()`를 빼면**: 생성자까지 기울기가 계산되지만 `optimizer_D`는 판별자 파라미터만 갱신하므로 **결과는 같다**. 다만 불필요한 기울기 계산으로 **느려지고 메모리를 더 쓴다**. 또 `optG.zero_grad()`를 빠뜨리면 생성자에 남은 기울기가 섞일 위험도 있다.

## 연습 11-6

LATENT_DIM의 값을 20, 200 등으로 바꿔 가며 학습하고, 100을 사용한 본문 예제와 비교해 생성 이미지의 다양성과 품질이 어떻게 달라지는지 살펴보자. 잠재 벡터의 차원이 너무 작거나 너무 크면 어떤 문제가 생기는지 함께 정리해 보자.

In [ ]:
for latent in (20, 100, 200):
    print(f'[LATENT_DIM={latent}]')
    G, _ = train_dcgan(latent_dim=latent, epochs=10)
    G.eval()
    with torch.no_grad():
        imgs = (G(torch.randn(16, latent, device=device)) + 1) / 2
    viz.plot_images(list(imgs.cpu()), [f'{latent}'] * 16, images_per_row=8)

**너무 작으면**(20) 표현할 수 있는 이미지의 가짓수가 제한되어 비슷한 이미지만 반복 생성된다(다양성 부족).

**너무 크면**(200) 잠재 공간이 넓어져 학습 데이터가 채우지 못한 영역이 늘고, 그 영역에서 뽑은 벡터는 흐릿하거나 깨진 이미지를 만든다. 학습도 불안정해진다.

Fashion-MNIST에서는 100 안팎이 무난하다.

## 연습 11-7

[연습 문제 11-4]에서 사용한 사람 얼굴로 구성된 컬러 이미지 데이터셋을 학습해서 무작위 컬러 얼굴 이미지를 생성하는 모델을 만들어 보자. 그리고 임의의 두 사람의 얼굴 이미지를 만든 후 [그림 11-5]와 같이 두 얼굴 이미지를 10단계로 보간하는 중간 이미지도 만들어 보자.

### 풀이

[연습 문제 11-4]의 컬러 얼굴 데이터셋을 DCGAN으로 학습하려면 다음을 바꾼다.

1. **출력 채널 1 → 3**, 이미지 64×64. 생성자는 `fc_z_to_fmap`이 (256, 8, 8)을 만들고 역합성곱 3번으로 8 → 16 → 32 → 64.
2. **판별자**도 대칭으로 합성곱을 하나 더 두어 64 → 32 → 16 → 8.
3. 정규화는 3채널 기준 `Normalize((0.5,)*3, (0.5,)*3)`.

**얼굴 보간**은 VAE와 달리 조금 다르다. GAN에는 이미지를 잠재 벡터로 되돌리는 인코더가 없으므로, **두 잠재 벡터 z₁, z₂를 먼저 뽑아** 각각의 얼굴을 생성한 뒤 그 사이를 보간한다.

```python
alphas = torch.linspace(0, 1, 10)
zs = torch.stack([(1 - a) * z1 + a * z2 for a in alphas])
images = G(zs)
```

부드럽게 변하면 잠재 공간이 의미 있게 학습된 것이다.

## 연습 11-8

[도전 문제] 11-1절 FashionVAE 클래스와 11-2절 생성자 Generator 클래스는 잠재 벡터를 역합성곱 계층에 입력하기 전 선형 계층(fc_z_to_fmap)을 사용해 LATENT_DIM 크기의 잠재 벡터를 합성곱 계층의 입력 특징 수에 맞춰 확장한다. 이를 다음과 같이 수정하고 결과를 분석해 보자.

입력 특징 수를 맞추는 계층으로 선형 계층 대신 역합성곱 계층을 사용하도록 FashionVAE 클래스와 Generator 클래스를 수정하고 결과를 확인해 보자.

두 구현 방식의 모델 구조와 생성 결과의 차이를 비교하고, 어떤 장단점이 있는지 정리해 보자.

In [ ]:
# 선형 계층 대신 역합성곱으로 잠재 벡터를 특징 지도로 펼친다.
class ConvGenerator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            # (B, latent, 1, 1) -> (B, 256, 7, 7)
            nn.ConvTranspose2d(latent_dim, 256, 7, 1, 0), nn.BatchNorm2d(256), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 4, 2, 1), nn.Tanh())
    def forward(self, z): return self.net(z.view(-1, self.latent_dim, 1, 1))

g = ConvGenerator()
print(f'출력 형태: {tuple(g(torch.randn(2, 100)).shape)}')
print(f'선형 방식 파라미터: {sum(p.numel() for p in Generator().parameters()):,}개')
print(f'역합성곱 방식     : {sum(p.numel() for p in g.parameters()):,}개')

**차이점**: 선형 계층은 잠재 벡터의 모든 성분을 특징 지도의 모든 위치에 연결하므로 파라미터가 매우 많다(100 × 12,544). 역합성곱은 커널을 공유해 파라미터가 훨씬 적다.

**장단점**: 역합성곱 방식은 파라미터가 적고 공간 구조를 처음부터 유지하지만, 초기 특징 지도가 커널 크기에 의해 결정되어 유연성이 떨어진다. 원조 DCGAN 논문은 역합성곱 방식을 사용한다.

## 연습 11-9

[도전 문제] 11-2절 마지막에 언급한 cGAN(조건부 GAN)은 손실 계산에 입력 이미지뿐 아니라 레이블도 사용하는 방식이다. 다음 두 단계로 나눠 cGAN에 도전해 보자.

11-2절의 DCGAN의 구조와 학습 방식 중 어떤 부분을 바꿔야 cGAN으로 동작하는지 직접 분석해 보자.

정리한 내용을 바탕으로 원하는 클래스의 이미지를 생성하는 모델을 만들어 보자. 클래스마다 이름이 있지만, 실제 생성은 0부터 9까지의 클래스 레이블로 지정하면 된다. 참고로 Fashion-MNIST의 클래스별 이름은 print(train_set.classes)로 출력해 볼 수 있다.

In [ ]:
# cGAN: 생성자와 판별자 모두에 클래스 레이블을 조건으로 입력
class CondGenerator(nn.Module):
    def __init__(self, latent_dim=100, n_class=10, embed=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.label_emb = nn.Embedding(n_class, embed)
        self.fc = nn.Sequential(
            nn.Linear(latent_dim + embed, 256 * 7 * 7),
            nn.BatchNorm1d(256 * 7 * 7), nn.ReLU())
        self.conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 4, 2, 1), nn.Tanh())
    def forward(self, z, y):
        h = self.fc(torch.cat([z, self.label_emb(y)], dim=1))
        return self.conv(h.view(-1, 256, 7, 7))

class CondDiscriminator(nn.Module):
    def __init__(self, n_class=10):
        super().__init__()
        self.label_map = nn.Embedding(n_class, 28 * 28)   # 레이블을 채널로 붙인다
        self.conv = nn.Sequential(
            nn.Conv2d(2, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128 * 7 * 7, 1), nn.Sigmoid())
    def forward(self, x, y):
        ymap = self.label_map(y).view(-1, 1, 28, 28)
        return self.fc(self.conv(torch.cat([x, ymap], dim=1)))

z = torch.randn(4, 100); y = torch.tensor([0, 1, 2, 3])
G, D = CondGenerator(), CondDiscriminator()
img = G(z, y)
print(f'조건부 생성 {tuple(img.shape)} / 판별 {tuple(D(img, y).shape)}')

In [ ]:
# 학습 루프는 레이블을 함께 넘기는 것만 다르다.
CLASSES = ['티셔츠/탑','바지','풀오버','드레스','코트','샌들','셔츠','스니커즈','가방','앵클부츠']
loader = fashion_loader(normalize_tanh=True)
torch.manual_seed(SEED)
G, D = CondGenerator().to(device), CondDiscriminator().to(device)
crit = nn.BCELoss()
optG = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
for e in range(1, 11):
    for real, y in loader:
        real, y = real.to(device), y.to(device); B = real.size(0)
        ones = torch.ones(B, 1, device=device); zeros = torch.zeros(B, 1, device=device)
        optD.zero_grad()
        z = torch.randn(B, 100, device=device)
        fake = G(z, y)
        (crit(D(real, y), ones) + crit(D(fake.detach(), y), zeros)).backward()
        optD.step()
        optG.zero_grad()
        z = torch.randn(B, 100, device=device)
        crit(D(G(z, y), y), ones).backward(); optG.step()
    if e % 5 == 0: print(f'{e}/10 에포크 완료')

G.eval()
with torch.no_grad():
    y = torch.arange(10, device=device)
    imgs = (G(torch.randn(10, 100, device=device), y) + 1) / 2
viz.plot_images(list(imgs.cpu()), CLASSES, images_per_row=5)

**cGAN으로 바꾸기 위해 필요한 변경**

1. **생성자**: 잠재 벡터에 레이블 임베딩을 이어 붙인다.
2. **판별자**: 레이블을 이미지와 같은 크기로 펼쳐 **채널로 추가**한다(입력 채널 1 → 2).
3. **학습 루프**: 진짜 이미지에는 진짜 레이블을, 가짜 이미지에는 생성에 사용한 레이블을 함께 넘긴다.

판별자가 '이 이미지가 진짜인가'뿐 아니라 '이 레이블에 맞는 이미지인가'까지 판단하므로, 생성자는 레이블에 맞는 이미지를 만들도록 학습된다.